> **Pre-configured: `Llama-3.1-8B-Instruct`, all 203 FOLIO validation examples, full precision.**  
> Recommended runtime: **L4 (24 GB) or A100**. Nothing needs editing — add `HF_TOKEN` to Secrets and Run all.

# NL-FOL · 1 · Generation

Translates FOLIO arguments from natural language into first-order logic with any Hugging Face model, and saves **everything** (raw output, parsed FOL, settings) to Google Drive.

**How to use**
1. Runtime → Change runtime type → **GPU**.
2. 🔑 Secrets panel → add `HF_TOKEN` (your Hugging Face token) → enable notebook access.
3. Edit **only Cell 3** (model + number of examples), then Runtime → **Run all**.

| Who | Cell 3 settings |
|---|---|
| Quick test | `NUM_EXAMPLES = 5` (on a free T4 also set `USE_4BIT = True`, see note below) |
| Full run | `NUM_EXAMPLES = None` (all 203) |
| Bigger models | change `MODEL`, e.g. `"Qwen3-32B"` or `"Llama-3.1-70B-Instruct"` (A100-80GB, or `USE_4BIT = True`) |

**Output** → `MyDrive/NL-FOL/runs/<RUN_NAME>.jsonl` (+ `.csv` and `_config.json`). Give `RUN_NAME` to the evaluation notebook.
If Colab disconnects, just **Run all** again; finished examples are skipped.

**GPU memory note:** a 7–8B model needs ~16 GB in 16-bit precision. A T4 has 15 GB, so part of the model is moved to CPU and generation becomes very slow (minutes per example). Use an L4 / A100 runtime, or set `USE_4BIT = True` for quick tests. 4-bit changes outputs slightly, so don't mix 4-bit and full-precision runs in one comparison. Cell 6 warns you if offloading happens.


## Cell 1 — Setup: install, GPU check, Google Drive

In [ ]:
%pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes datasets huggingface_hub pandas

import os, json, gc, re, time
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name} | {_p.total_memory/1e9:.1f} GB")

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NL-FOL"
RUNS_DIR    = f"{PROJECT_DIR}/runs"
os.makedirs(RUNS_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)


## Cell 2 — Hugging Face login (token from Colab Secrets)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    _tok = userdata.get("HF_TOKEN")
except Exception as e:
    _tok = None
    print("HF_TOKEN secret not found:", e)
login(token=_tok) if _tok else login()


## Cell 3 — Configuration ✏️ (the only cell to edit)
`MODEL` can be a short name from the list below **or** any Hugging Face path (e.g. `"mistralai/Mistral-7B-Instruct-v0.3"`).

In [ ]:
MODELS = {
    "Llama-3.1-8B-Instruct":  "meta-llama/Llama-3.1-8B-Instruct",
    "Llama-3.1-70B-Instruct": "meta-llama/Llama-3.1-70B-Instruct",
    "Qwen2.5-7B-Instruct":    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen2.5-72B-Instruct":   "Qwen/Qwen2.5-72B-Instruct",
    "Qwen3-32B":              "Qwen/Qwen3-32B",
}

MODEL        = "Llama-3.1-8B-Instruct"   # <- change model here
NUM_EXAMPLES = None                    # 5 = quick test, None = full FOLIO validation (203)
USE_4BIT     = False                   # True for big models on smaller GPUs (report it if used)

# "original" = settings of all earlier runs (T=0.1 sampling, repetition_penalty 1.15)
# "greedy"   = deterministic decoding, no repetition penalty (recommended for new experiments)
GEN_PRESET     = "original"
PROMPT_VERSION = "logicllama_v1"
SEED           = 42
MAX_NEW_TOKENS = 1024

RUN_NAME = None    # None = automatic, e.g. "Qwen2.5-7B-Instruct__logicllama_v1__original__n5"

# ---------------------------------------------------------------------------
HF_PATH    = MODELS.get(MODEL, MODEL)
MODEL_NAME = MODEL if MODEL in MODELS else MODEL.split("/")[-1]
GEN_PRESETS = {
    "original": dict(do_sample=True,  temperature=0.1, repetition_penalty=1.15),
    "greedy":   dict(do_sample=False, temperature=None, repetition_penalty=1.0),
}
GEN_KWARGS = GEN_PRESETS[GEN_PRESET]
_n = "all" if NUM_EXAMPLES is None else NUM_EXAMPLES
RUN_NAME = RUN_NAME or f"{MODEL_NAME}__{PROMPT_VERSION}__{GEN_PRESET}__n{_n}"
CHECKPOINT = f"{RUNS_DIR}/{RUN_NAME}.jsonl"
print(f"Model : {MODEL_NAME}  ({HF_PATH})\nRun   : {RUN_NAME}\nFile  : {CHECKPOINT}")


## Cell 4 — Functions (prompt, parsers, model runner)
No need to edit. Contents:
* **Prompt**: LogicLLaMA-style system prompt + 5 few-shot examples. Identical to the Qwen2.5 / Llama-3.1 / Qwen3 runs.
* **Parsers**: `parse_fol_output_strict` (original) and `parse_fol_output_lenient` (also accepts markdown headers like `**PREMISES_FOL:**`, which Llama produces). The lenient parser ignores formatting only and never edits formulas. Both results are saved.
* **Runner**: uses each model's official chat template (one BOS token) and stops on all of its end tokens (e.g. Llama's `<|eot_id|>`).

In [ ]:
# ════════════════════ PROMPT (logicllama_v1) ════════════════════
# ── System prompt — enforces LogicLLaMA FOL notation rules ────────────
SYSTEM_PROMPT = """You are a logic expert. Translate natural language statements into \
first-order logic (FOL) rules.

FOL NOTATION RULES — follow exactly:
1. You MUST USE these logical operators: ⊕ (either or), ∨ (disjunction), ∧ (conjunction), \
   → (implication), ∀ (universal), ∃ (existential), ¬ (negation), ↔ (equivalence)
2. You MUST NEVER USE: =, !=, %, bare variable expressions like "y = a ∨ y = b"
3. Literals MUST always have a predicate name and entity, e.g. Rounded(x) or City(guilin)
4. ALWAYS put quantifiers and variables at the beginning of the FOL formula
5. Use only these variable sets: no variables; one variable x; two variables x,y; \
   three variables x,y,z
6. AVOID long predicate names — use short, clear names like Performs(x) not \
   PerformsInSchoolTalentShowOften(x)
7. The FOL MUST accurately reflect the meaning of the NL statement
"""

# ── Few-shot examples from MALLS / Table 5 of Yang et al. 2024 ────────
FEW_SHOT_EXAMPLES = """Examples of correct NL → FOL translation:

NL: If someone is entire, then he is not serious, and vice versa.
FOL: ∃x Entire(x) ↔ ¬Serious(x)

NL: If there is at least one person who is both not excited and not timid, then Jonathan is elderly.
FOL: (∃x ¬Excited(x) ∧ ¬Timid(x)) → Elderly(Jonathan)

NL: Someone who is either not fresh or entire is always not serious.
FOL: ∀x (¬Fresh(x) ∨ Entire(x)) → ¬Serious(x)

NL: If Nathalie is not blue, then Collier is entire.
FOL: ¬Blue(Nathalie) → Entire(Collier)

NL: Someone is courteous and not elderly if and only if he is not excited and not various.
FOL: ∃x (Courteous(x) ∧ ¬Elderly(x)) ↔ (¬Excited(x) ∧ ¬Various(x))
"""

def build_fol_prompt(premises, conclusion):
    pstr = "\n".join(f"{i+1}. {p}" for i, p in enumerate(premises))
    return (
        f"{FEW_SHOT_EXAMPLES}\n"
        "Now translate the following argument. "
        "Follow the FOL notation rules exactly.\n\n"
        "=== INPUT ===\n"
        f"PREMISES:\n{pstr}\n\n"
        f"CONCLUSION:\n{conclusion}\n\n"
        "=== OUTPUT FORMAT ===\n"
        "Respond using ONLY these four labeled sections:\n\n"
        "ENTITIES: entity1, entity2, ...\n\n"
        "PREDICATES:\n"
        "- PredicateName(x): short meaning\n\n"
        "PREMISES_FOL:\n"
        "1. <FOL formula>\n"
        "2. <FOL formula>\n\n"
        "CONCLUSION_FOL: <FOL formula>\n"
    )


def parse_premises(raw):
    if isinstance(raw, list):
        return [p.strip() for p in raw if p.strip()]
    return [p.strip() for p in raw.split("\n") if p.strip()]


# ════════════════════ PARSERS ════════════════════

# ── Original parser (verbatim) ────────────────────────────────────────────
def parse_fol_output_strict(raw_text):
    result = {"entities": "", "predicates": "", "premises_fol": [],
              "conclusion_fol": "", "parse_success": False, "raw_output": raw_text.strip()}

    m = re.search(r"ENTITIES:\s*(.+?)(?:\n|$)", raw_text, re.IGNORECASE)
    if m: result["entities"] = m.group(1).strip()

    m = re.search(r"PREDICATES:\s*\n(.*?)(?=PREMISES_FOL:|STEP 3|$)", raw_text, re.IGNORECASE | re.DOTALL)
    if m: result["predicates"] = m.group(1).strip()

    m = re.search(r"PREMISES_FOL:\s*\n(.*?)(?=CONCLUSION_FOL:|STEP 4|$)", raw_text, re.IGNORECASE | re.DOTALL)
    if m:
        lines   = [l.strip() for l in m.group(1).strip().split("\n") if l.strip()]
        cleaned = [re.sub(r"^\d+[.)\s]+", "", l).strip() for l in lines]
        result["premises_fol"] = [c for c in cleaned if c]

    m = re.search(r"CONCLUSION_FOL:\s*(.+?)(?:\n|$)", raw_text, re.IGNORECASE)
    if m: result["conclusion_fol"] = m.group(1).strip()

    if result["premises_fol"] and result["conclusion_fol"]:
        result["parse_success"] = True

    if result["conclusion_fol"]:
        result["conclusion_fol"] = re.sub(r"^[\*\-\s]+", "", result["conclusion_fol"]).strip()

    return result


# ── Lenient parser ────────────────────────────────────────────────────────
_HEADER_PATTERNS = [
    ("entities",   r"ENTITIES"),
    ("predicates", r"PREDICATES"),
    ("premises",   r"PREMIS+ES?[\s_\-]*\(?\s*FOL\s*\)?"),
    ("conclusion", r"CONCLUSIONS?[\s_\-]*\(?\s*FOL\s*\)?"),
]

# A header must start a line, optionally preceded by markdown/list/step noise.
_LINE_PREFIX = r"^[ \t>#\-\*\d\.\)]*(?:step\s*\d+\s*[:.\-–]?\s*)?"


def _strip_markdown(text):
    text = re.sub(r"```[a-zA-Z]*", "", text)       # code fences
    text = text.replace("**", "").replace("__", "")
    text = text.replace("`", "")
    return text


def _find_headers(text):
    """Return sorted list of (start, content_start, name)."""
    hits = []
    for name, pat in _HEADER_PATTERNS:
        rx = re.compile(_LINE_PREFIX + r"(" + pat + r")\s*:?", re.IGNORECASE | re.MULTILINE)
        for m in rx.finditer(text):
            hits.append((m.start(), m.end(), name))
    hits.sort()
    # keep the first occurrence of each section (models sometimes echo the format)
    seen, out = set(), []
    for h in hits:
        if h[2] in seen:
            continue
        seen.add(h[2])
        out.append(h)
    return sorted(out)


def _section_bodies(text):
    headers = _find_headers(text)
    bodies = {}
    for i, (start, cstart, name) in enumerate(headers):
        end = headers[i + 1][0] if i + 1 < len(headers) else len(text)
        bodies[name] = text[cstart:end]
    return bodies


def _clean_formula_line(line):
    line = re.sub(r"^\s*[\-\*•]?\s*", "", line)          # bullets
    line = re.sub(r"^\(?\d+[.)]\s*", "", line)            # numbering "1." "1)" "(1)"
    line = re.sub(r"\s*//.*$", "", line)                  # trailing // comment
    return line.strip()


def _is_comment(line):
    return bool(re.match(r"^\s*(note|explanation|where|here)\b", line, re.IGNORECASE))


def parse_fol_output_lenient(raw_text):
    result = {"entities": "", "predicates": "", "premises_fol": [],
              "conclusion_fol": "", "parse_success": False,
              "raw_output": raw_text.strip() if isinstance(raw_text, str) else ""}
    if not isinstance(raw_text, str) or not raw_text.strip():
        return result

    text = _strip_markdown(raw_text)
    bodies = _section_bodies(text)

    if "entities" in bodies:
        ents = " ".join(l.strip() for l in bodies["entities"].splitlines() if l.strip())
        result["entities"] = ents

    if "predicates" in bodies:
        result["predicates"] = bodies["predicates"].strip()

    if "premises" in bodies:
        prem = []
        for line in bodies["premises"].splitlines():
            if not line.strip() or _is_comment(line):
                continue
            c = _clean_formula_line(line)
            if c and "(" in c:              # must contain at least one predicate call
                prem.append(c)
            elif c and prem and re.match(r"^[∧∨→↔⊕¬(]", c):
                prem[-1] = prem[-1] + " " + c   # continuation line
        result["premises_fol"] = prem

    if "conclusion" in bodies:
        lines, started = [], False
        for line in bodies["conclusion"].splitlines():
            if not line.strip():
                if started:
                    break
                continue
            if _is_comment(line):
                break
            c = _clean_formula_line(line)
            if c:
                lines.append(c)
                started = True
        concl = " ".join(lines).strip()
        result["conclusion_fol"] = concl if "(" in concl else ""

    result["parse_success"] = bool(result["premises_fol"] and result["conclusion_fol"])
    return result


# ════════════════════ MODEL RUNNER ════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

def load_model(hf_path, use_4bit=False):
    tok = AutoTokenizer.from_pretrained(hf_path)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4 has no bf16
    kw = {"device_map": "auto", "dtype": dtype}
    if use_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    model = AutoModelForCausalLM.from_pretrained(hf_path, **kw).eval()
    offloaded = [k for k, v in getattr(model, "hf_device_map", {}).items() if v in ("cpu", "disk")]
    if offloaded:
        print(f"WARNING: {len(offloaded)} model parts don't fit on the GPU and run on CPU -> very slow.\n"
              "         Use a bigger GPU runtime, or set USE_4BIT = True in Cell 3.")
    return tok, model

def stop_token_ids(tok, model):
    ids = set()
    for v in (tok.eos_token_id, getattr(model.generation_config, "eos_token_id", None)):
        if isinstance(v, int): ids.add(v)
        elif isinstance(v, (list, tuple)): ids.update(i for i in v if isinstance(i, int))
    for t in ("<|eot_id|>", "<|end_of_text|>", "<|im_end|>", "<|endoftext|>"):
        tid = tok.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid != tok.unk_token_id:
            ids.add(tid)
    return sorted(ids)

def format_prompt(tok, system, user):
    # enable_thinking=False is read by Qwen3's template and ignored by other models
    return tok.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

def generate(model, tok, prompt_text, stop_ids, max_new_tokens, gen_kwargs):
    enc = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    kw = {k: v for k, v in gen_kwargs.items() if v is not None}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             eos_token_id=stop_ids, pad_token_id=stop_ids[0], **kw)
    new = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False), int(len(new))

def read_jsonl(path):
    rows = []
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: pass    # half-written line after a disconnect
    return rows

_t = "**ENTITIES:** Tom\n\n**PREMISES_FOL:**\n1. P(Tom)\n\n**CONCLUSION_FOL:** Q(Tom)"
assert parse_fol_output_lenient(_t)["parse_success"] and not parse_fol_output_strict(_t)["parse_success"]
print("Functions ready.")


## Cell 5 — Load FOLIO validation set

In [ ]:
from datasets import load_dataset

folio = load_dataset("yale-nlp/folio", split="validation")
n = len(folio) if NUM_EXAMPLES is None else min(NUM_EXAMPLES, len(folio))
examples = folio.select(range(n))
print(f"Using {n} of {len(folio)} examples")


## Cell 6 — Run generation (saved to Drive after every example)

In [ ]:
done = {r["example_id"] for r in read_jsonl(CHECKPOINT)}
todo = [i for i in range(len(examples)) if i not in done]
print(f"Already done: {len(done)} | to run: {len(todo)}")

config = dict(run_name=RUN_NAME, model=MODEL_NAME, hf_path=HF_PATH, prompt_version=PROMPT_VERSION,
              gen_preset=GEN_PRESET, gen_kwargs=GEN_KWARGS, max_new_tokens=MAX_NEW_TOKENS,
              seed=SEED, use_4bit=USE_4BIT, num_examples=len(examples),
              gpu=torch.cuda.get_device_name(0), started=time.strftime("%Y-%m-%d %H:%M"))
json.dump(config, open(f"{RUNS_DIR}/{RUN_NAME}_config.json", "w"), indent=2)

if todo:
    print(f"Loading {HF_PATH} ...")
    tok, model = load_model(HF_PATH, USE_4BIT)
    stops = stop_token_ids(tok, model)
    _probe = format_prompt(tok, SYSTEM_PROMPT, "x")
    print("Stop token ids:", stops, "| BOS count:", _probe.count(tok.bos_token) if tok.bos_token else "n/a")

    t0 = time.time()
    with open(CHECKPOINT, "a", encoding="utf-8") as fout:
        for k, idx in enumerate(todo, 1):
            ex = examples[idx]
            premises = parse_premises(ex["premises"])
            set_seed(SEED + idx)
            raw, n_new = generate(model, tok,
                                  format_prompt(tok, SYSTEM_PROMPT, build_fol_prompt(premises, ex["conclusion"])),
                                  stops, MAX_NEW_TOKENS, GEN_KWARGS)
            s, l = parse_fol_output_strict(raw), parse_fol_output_lenient(raw)
            row = {
                "run_name": RUN_NAME, "model": MODEL_NAME, "example_id": idx,
                "nl_premises": premises, "nl_conclusion": ex["conclusion"], "gold_label": ex["label"],
                "gold_premises_fol": ex["premises-FOL"], "gold_conclusion_fol": ex["conclusion-FOL"],
                "entities": l["entities"], "predicates": l["predicates"],
                "premises_fol": l["premises_fol"], "conclusion_fol": l["conclusion_fol"],
                "parse_success": l["parse_success"], "parse_success_strict": s["parse_success"],
                "n_new_tokens": n_new, "hit_token_limit": n_new >= MAX_NEW_TOKENS,
                "raw_output": raw,
            }
            fout.write(json.dumps(row, ensure_ascii=False) + "\n"); fout.flush()
            eta = (time.time() - t0) / k * (len(todo) - k) / 60
            print(f"[{idx:3d}] strict={'OK ' if s['parse_success'] else 'FAIL'} "
                  f"lenient={'OK ' if l['parse_success'] else 'FAIL'} tok={n_new:4d} "
                  f"eta={eta:5.1f}m | {l['conclusion_fol'][:50]}")

    del model, tok; gc.collect(); torch.cuda.empty_cache()
print("Generation finished.")


## Cell 7 — Export CSV + summary

In [ ]:
import pandas as pd

df = (pd.DataFrame(read_jsonl(CHECKPOINT)).drop_duplicates("example_id", keep="last")
        .sort_values("example_id").reset_index(drop=True))
csv_df = df.copy()
for col in ("nl_premises", "premises_fol"):
    csv_df[col] = csv_df[col].apply(lambda v: " | ".join(v) if isinstance(v, list) else v)
csv_df.to_csv(f"{RUNS_DIR}/{RUN_NAME}.csv", index=False)

def failure_reason(r):
    raw = r["raw_output"] or ""
    if not raw.strip():                 return "empty output"
    if r["hit_token_limit"]:            return "hit token limit"
    if "PREMIS" not in raw.upper():     return "no premises section"
    if "CONCLUSION" not in raw.upper(): return "no conclusion section"
    return "sections present, no formula parsed"

print(f"Run: {RUN_NAME}  ({len(df)} examples)")
print(f"  parse rate, original parser : {df.parse_success_strict.mean()*100:5.1f}%")
print(f"  parse rate, lenient parser  : {df.parse_success.mean()*100:5.1f}%")
print(f"  hit token limit             : {int(df.hit_token_limit.sum())}")
print(f"  mean new tokens             : {df.n_new_tokens.mean():.0f}")
fails = df[~df.parse_success]
if len(fails):
    print("\nFailures:"); print(fails.apply(failure_reason, axis=1).value_counts().to_string())
print(f"\nSaved to {RUNS_DIR}/  ->  {RUN_NAME}.jsonl / .csv / _config.json")
print(f'For the evaluation notebook: RUN_NAME = "{RUN_NAME}"')


## Cell 8 — Inspect raw output (one example, or the failures)

In [ ]:
INSPECT_EXAMPLE = 0      # an example id, or None to print failures
SHOW_FAILURES   = 3

def show(r):
    print(f"\n===== example {r.example_id} | strict={r.parse_success_strict} lenient={r.parse_success} | tokens={r.n_new_tokens} =====")
    print("NL conclusion :", r.nl_conclusion, "| gold label:", r.gold_label)
    print("Conclusion FOL:", r.conclusion_fol)
    print("Gold conclusion:", r.gold_conclusion_fol)
    print("----- raw output -----\n" + r.raw_output[:2000])

if INSPECT_EXAMPLE is not None:
    show(df[df.example_id == INSPECT_EXAMPLE].iloc[0])
else:
    for _, r in fails.head(SHOW_FAILURES).iterrows(): show(r)
